In [0]:
# ============================================================
# NOTEBOOK 05 — APLICACIÓN: DASHBOARD
# Vermont School - Sistema de Alerta Temprana 2025-26
# ============================================================
%pip install plotly

In [0]:
# CELDA 2 — Cargar datos para dashboard
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

# Cargar datos
df = spark.read.parquet(f"{TRUSTED}/estudiantes_2025_26").toPandas()
df_pred = spark.read.parquet(f"{SILVER}/modelo_predicciones").toPandas()
df_asig = spark.read.parquet(f"{SILVER}/eda_riesgo_por_asignatura").toPandas()
df_riesgo_grado = spark.read.parquet(f"{SILVER}/eda_riesgo_por_grado").toPandas()

# Mapeo de etiquetas para visualización
label_map = {0: 'Riesgo Crítico', 1: 'Riesgo Recuperación', 2: 'Sin Riesgo'}
df_pred['nivel_predicho'] = df_pred['prediction'].map(label_map)
df_pred['nivel_real']     = df_pred['nivel_riesgo'].map({
    'riesgo_critico':      'Riesgo Crítico',
    'riesgo_recuperacion': 'Riesgo Recuperación',
    'sin_riesgo':          'Sin Riesgo'
})

colores = {
    'Riesgo Crítico':      '#e74c3c',
    'Riesgo Recuperación': '#f39c12',
    'Sin Riesgo':          '#2ecc71'
}

print(f"✓ Datos cargados para dashboard")
print(f"  Estudiantes: {len(df)}")
print(f"  Predicciones: {len(df_pred)}")

In [0]:
# CELDA 3 — Dashboard completo

# ── GRÁFICO 1: Distribución de riesgo por grado ──
fig1 = px.bar(
    df_riesgo_grado,
    x='grado', y='n_estudiantes',
    color='nivel_riesgo',
    barmode='group',
    color_discrete_map={
        'riesgo_critico':      '#e74c3c',
        'riesgo_recuperacion': '#f39c12',
        'sin_riesgo':          '#2ecc71'
    },
    title='Distribución de riesgo por grado — Vermont School 2025-26',
    labels={'grado': 'Grado', 'n_estudiantes': 'N° Estudiantes',
            'nivel_riesgo': 'Nivel de Riesgo'},
    text='n_estudiantes'
)
fig1.update_traces(textposition='outside')
fig1.update_layout(xaxis=dict(tickvals=['7','8','9'],
                              ticktext=['7°','8°','9°']))
fig1.show()

# ── GRÁFICO 2: Asignaturas con más estudiantes bajo 4.0 ──
fig2 = px.bar(
    df_asig.sort_values('n_bajo_4', ascending=True),
    x='n_bajo_4', y='asignatura',
    orientation='h',
    color='n_bajo_4',
    color_continuous_scale='Reds',
    title='Estudiantes bajo 4.0 por asignatura',
    labels={'n_bajo_4': 'N° estudiantes bajo 4.0',
            'asignatura': 'Asignatura'},
    text='n_bajo_4'
)
fig2.update_traces(textposition='outside')
fig2.show()

# ── GRÁFICO 3: Ausencias vs Promedio (scatter) ──
df['nivel_label'] = df['nivel_riesgo'].map({
    'riesgo_critico':      'Riesgo Crítico',
    'riesgo_recuperacion': 'Riesgo Recuperación',
    'sin_riesgo':          'Sin Riesgo'
})
fig3 = px.scatter(
    df,
    x='total_ausencias',
    y='promedio_acumulado',
    color='nivel_label',
    color_discrete_map=colores,
    size='n_f1',
    hover_data=['grado', 'seccion_anon', 'n_f1', 'n_f2'],
    title='Ausencias vs Promedio académico<br>(tamaño = N° de F1)',
    labels={
        'total_ausencias':    'Total ausencias',
        'promedio_acumulado': 'Promedio acumulado',
        'nivel_label':        'Nivel de riesgo'
    }
)
fig3.add_hline(y=4.0, line_dash="dash", line_color="red",
               annotation_text="Límite aprobación (4.0)")
fig3.show()

# ── GRÁFICO 4: Matriz de confusión del modelo ──
from sklearn.metrics import confusion_matrix
import numpy as np

orden = ['Riesgo Crítico', 'Riesgo Recuperación', 'Sin Riesgo']
cm = confusion_matrix(
    df_pred['nivel_real'],
    df_pred['nivel_predicho'],
    labels=orden
)

fig4 = px.imshow(
    cm,
    labels=dict(x="Predicho", y="Real", color="N°"),
    x=orden, y=orden,
    color_continuous_scale='Blues',
    title='Matriz de confusión — Random Forest',
    text_auto=True
)
fig4.show()

# ── GRÁFICO 5: Top 10 estudiantes en riesgo crítico ──
df_criticos = df[df['nivel_riesgo'] == 'riesgo_critico']\
    .sort_values('promedio_acumulado')\
    .head(10)[['id_estudiante','grado','seccion_anon',
               'promedio_acumulado','total_ausencias','n_f1','n_f2']]

fig5 = go.Figure(data=[go.Table(
    header=dict(
        values=['ID', 'Grado', 'Sección', 'Promedio', 'Ausencias', 'F1', 'F2'],
        fill_color='#e74c3c',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[
            df_criticos['id_estudiante'],
            df_criticos['grado'],
            df_criticos['seccion_anon'],
            df_criticos['promedio_acumulado'].round(2),
            df_criticos['total_ausencias'],
            df_criticos['n_f1'],
            df_criticos['n_f2']
        ],
        fill_color=[['#fadbd8' if i%2==0 else 'white'
                     for i in range(len(df_criticos))]],
        align='center'
    )
)])
fig5.update_layout(title='Top 10 estudiantes en riesgo crítico')
fig5.show()

print("✓ Dashboard completo — 5 visualizaciones generadas")

In [0]:
# CELDA 4 — Guardar resumen final en Silver
import json
from datetime import datetime

resumen = {
    "proyecto": "Vermont School - Sistema de Alerta Temprana",
    "fecha": datetime.now().strftime('%Y-%m-%d %H:%M'),
    "año_lectivo": "2025-2026",
    "pipeline": {
        "bronze": {
            "archivos": 4,
            "formato": "XLS",
            "fuente": "Phidias"
        },
        "trusted": {
            "estudiantes": 149,
            "columnas": 71,
            "formato": "Parquet",
            "anonimizado": True
        },
        "silver": {
            "tablas_eda": 4,
            "modelo": "Random Forest",
            "f1_score_cv": 0.8804,
            "mejores_hiperparametros": {
                "numTrees": 50,
                "maxDepth": 3
            }
        }
    },
    "hallazgos": {
        "estudiantes_sin_riesgo": 81,
        "estudiantes_riesgo_recuperacion": 49,
        "estudiantes_riesgo_critico": 19,
        "asignatura_mas_reprobada": "Mathematics (38 estudiantes bajo 4.0)",
        "variable_mas_importante": "nota_min_acumulada (0.50)",
        "promedio_ausencias_critico": 72.5,
        "promedio_ausencias_sin_riesgo": 40.9
    }
}

# Guardar como JSON en Silver
ruta_resumen = f"{SILVER}/resumen_pipeline.json"
with open(ruta_resumen.replace('/Volumes', '/Volumes'), 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

# También como tabla Spark
resumen_df = spark.createDataFrame([{
    'fecha': resumen['fecha'],
    'estudiantes_total': 149,
    'sin_riesgo': 81,
    'riesgo_recuperacion': 49,
    'riesgo_critico': 19,
    'f1_score': 0.8804,
    'variable_top': 'nota_min_acumulada'
}])
resumen_df.write.mode("overwrite").parquet(f"{SILVER}/resumen_final")

print("=" * 55)
print("PIPELINE COMPLETO — Vermont School 2025-26")
print("=" * 55)
print(f"✓ Notebook 01 — Bronze: ingesta XLS")
print(f"✓ Notebook 02 — Trusted: preparación y anonimización")
print(f"✓ Notebook 03 — Silver: EDA con SparkSQL")
print(f"✓ Notebook 04 — Silver: modelo Random Forest")
print(f"✓ Notebook 05 — Aplicación: dashboard")
print(f"\nF1-Score (CV k=5): 0.8804")
print(f"Estudiantes en riesgo crítico: 19 (12.8%)")
print(f"Asignatura crítica: Mathematics")
print(f"Variable más predictiva: nota_min_acumulada")
print(f"\n✓ Todos los resultados guardados en Silver")